In [1]:
import numpy as np
from firedrake import *
from firedrake.cython import dmcommon
from petsc4py import PETSc

firedrake:WARNING OMP_NUM_THREADS is not set or is set to a value greater than 1, we suggest setting OMP_NUM_THREADS=1 to improve performance


In [2]:
def solve_poisson(nref, degree):
    dim = 2
    label_ext = 1
    label_interf = 2
    distribution_parameters_noop = {
        "partition": True,
        "overlap_type": (DistributedMeshOverlapType.NONE, 0),
    }
    mesh = Mesh("/home/charlottecai/MSc_Project_new/my_mix_mesh.msh", distribution_parameters=distribution_parameters_noop)
    plex = mesh.topology_dm
    for _ in range(nref):
        plex = plex.refine()
    plex.removeLabel("pyop2_core")
    plex.removeLabel("pyop2_owned")
    plex.removeLabel("pyop2_ghost")
    mesh = Mesh(plex)
    h = 0.1 / 2**nref  # roughly
    mesh.topology_dm.markBoundaryFaces(dmcommon.FACE_SETS_LABEL, label_ext)

# Extract submesh
    # triangle mesh
    mesh_t = Submesh(mesh, dim, PETSc.DM.PolytopeType.TRIANGLE, label_name="celltype", name="mesh_tri")
    x_t, y_t = SpatialCoordinate(mesh_t)
    n_t = FacetNormal(mesh_t)

    # quadrilateral mesh
    mesh_q = Submesh(mesh, dim, PETSc.DM.PolytopeType.QUADRILATERAL, label_name="celltype", name="mesh_quad")
    x_q, y_q = SpatialCoordinate(mesh_q)
    n_q = FacetNormal(mesh_q)

# Define function space and mixed function space
    V_t = FunctionSpace(mesh_t, "P", degree)
    V_q = FunctionSpace(mesh_q, "Q", degree)
    V = V_t * V_q

    u = TrialFunction(V)
    v = TestFunction(V)

    u_t, u_q = split(u)
    v_t, v_q = split(v)

# Define integral measure & interface measure between triangle and quadrilateral mesh
    dx_t = Measure("dx", mesh_t)
    dx_q = Measure("dx", mesh_q)
    ds_t = Measure("ds", mesh_t, intersect_measures=(Measure("ds", mesh_q),))
    ds_q = Measure("ds", mesh_q, intersect_measures=(Measure("ds", mesh_t),))

# Manufactured solution
    g_t = cos(2 * pi * x_t) * cos(2 * pi * y_t)
    g_q = cos(2 * pi * x_q) * cos(2 * pi * y_q)
    f_t = 8 * pi**2 * g_t
    f_q = 8 * pi**2 * g_q

    a = (
        inner(grad(u_t), grad(v_t)) * dx_t + inner(grad(u_q), grad(v_q)) * dx_q
        - inner(
            (grad(u_q) + grad(u_t)) / 2,
            (v_q * n_q + v_t * n_t)
        ) * ds_q(label_interf)
        - inner(
            (u_q * n_q + u_t * n_t),
            (grad(v_q) + grad(v_t)) / 2
        ) * ds_t(label_interf)
        + 100 / h * inner(u_q - u_t, v_q - v_t) * ds_q(label_interf)
    )
    L = (
        inner(f_t, v_t) * dx_t + inner(f_q, v_q) * dx_q
    )

# Dirichlet boundary conditions    
    sol = Function(V)
    bc_t = DirichletBC(V.sub(0), g_t, label_ext)
    bc_q = DirichletBC(V.sub(1), g_q, label_ext)
    solve(a == L, sol, bcs=[bc_t, bc_q])
    sol_t, sol_q = split(sol)

# L2 error
    L2Error_t = assemble(inner(sol_t - g_t, sol_t - g_t) * dx_t)
    L2Error_q = assemble(inner(sol_q - g_q, sol_q - g_q) * dx_q)
# H1 error
    H1Error_t = L2Error_t + assemble(inner(grad(sol_t - g_t), grad(sol_t - g_t)) * dx_t)
    H1Error_q = L2Error_q + assemble(inner(grad(sol_q - g_q), grad(sol_q - g_q)) * dx_q)
    return sqrt(L2Error_t + L2Error_q), sqrt(H1Error_t + H1Error_q)

In [3]:
# Convergence Rate Analysis
for degree in [1, 2, 3, 4]:
    PETSc.Sys.Print("degree = ", degree)
    L2Errors = []
    H1Errors = []
    for nref in range(4):
        PETSc.Sys.Print("nref = ", nref)
        L2Error, H1Error = solve_poisson(nref, degree)
        L2Errors.append(L2Error)
        H1Errors.append(H1Error)
    PETSc.Sys.Print(np.log2(L2Errors))
    PETSc.Sys.Print(np.log2(H1Errors))
    L2Errors = [np.log2(c) - np.log2(f) for c, f in zip(L2Errors[:-1], L2Errors[1:])]
    H1Errors = [np.log2(c) - np.log2(f) for c, f in zip(H1Errors[:-1], H1Errors[1:])]
    PETSc.Sys.Print(L2Errors)
    PETSc.Sys.Print(H1Errors)

degree =  1
nref =  0
nref =  1
nref =  2
nref =  3
[ -4.69501068  -6.67576617  -8.66967189 -10.66787722]
[ 0.38072356 -0.61225447 -1.60966766 -2.60885205]
[np.float64(1.9807554941717607), np.float64(1.9939057246975604), np.float64(1.9982053271701883)]
[np.float64(0.9929780299953794), np.float64(0.9974131909141657), np.float64(0.9991843864340322)]
degree =  2
nref =  0
nref =  1
nref =  2
nref =  3
[ -9.19331088 -12.18209807 -15.17969813 -18.17907589]
[-3.01459845 -5.00619337 -7.00337488 -9.00215584]
[np.float64(2.9887871879176213), np.float64(2.9976000579930506), np.float64(2.9993777608815737)]
[np.float64(1.991594914218818), np.float64(1.9971815169218061), np.float64(1.9987809551429496)]
degree =  3
nref =  0
nref =  1
nref =  2
nref =  3
[-13.68276324 -17.68638535 -21.69142483 -25.69536819]
[ -6.99805424  -9.99614184 -12.99586494 -15.99593395]
[np.float64(4.003622106616957), np.float64(4.00503948435685), np.float64(4.003943351782585)]
[np.float64(2.998087597362951), np.float64(2.999